In [ ]:
import numpy as np
import pandas as pd
from google.colab import files
activity = pd.read_csv('customer_flight_activity_cleaned.csv')
loyalty = pd.read_csv('customer_loyalty_history_cleaned.csv')

print('activity:', activity.shape)
print('loyalty:', loyalty.shape)
activity.head(3)

activity: (391014, 7)
loyalty: (16737, 16)


,Loyalty Number,Year,Month,Total Flights,Distance,Points Accumulated,Points Redeemed
0,100590,2018,6,12,15276,22914.0,0
1,100590,2018,7,12,9168,13752.0,0
2,100590,2018,5,4,6504,9756.0,0


## 1. Choosing the cutoff

The flight activity data covers **Jan 2017 – Dec 2018** (24 months). To avoid data leakage, we split this timeframe into two separate windows :

- **Feature window**: Jan 2017 → Jun 2018 (18 months of history)
- **Label window**: Jul 2018 → Dec 2018 (6 months to observe the outcome)


In [ ]:
activity['Date'] = pd.to_datetime(
    pd.DataFrame({'year': activity['Year'], 'month': activity['Month'], 'day': 1})
)

CUTOFF = pd.Timestamp('2018-06-01')
LABEL_END = pd.Timestamp('2018-12-01')

feat_activity = activity[activity['Date'] <= CUTOFF].sort_values(['Loyalty Number', 'Date'])
label_activity = activity[(activity['Date'] > CUTOFF) & (activity['Date'] <= LABEL_END)]

print('feature window rows:', len(feat_activity))
print('label window rows:', len(label_activity))

feature window rows: 289999
label window rows: 101015


## 2. Behavioural features (feature window only)

Everything below is computed only using the data upto and
including the cutoff. None of it looks into the label window.


In [ ]:
grp = feat_activity.groupby('Loyalty Number')

behaviour = pd.DataFrame({
    'Total Flights in Lifetime': grp['Total Flights'].sum(),
    'Total Distance': grp['Distance'].sum(),
    'Net Reward Points': grp['Points Accumulated'].sum() - grp['Points Redeemed'].sum(),
})

earned = grp['Points Accumulated'].sum()
redeemed = grp['Points Redeemed'].sum()
behaviour['Redeem Ratio'] = np.where(earned == 0, 0, round(redeemed / earned, 3))

behaviour = behaviour.reset_index()
behaviour.head()

,Loyalty Number,Total Flights in Lifetime,Total Distance,Net Reward Points,Redeem Ratio
0,100018,29,44813,43685.0,0.025
1,100102,34,50123,48928.0,0.024
2,100140,36,52368,51775.0,0.011
3,100214,13,23915,23054.0,0.036
4,100272,28,38585,37578.0,0.026


In [ ]:

last_flight_date = (
    feat_activity[feat_activity['Total Flights'] > 0]
    .groupby('Loyalty Number')['Date'].max()
)

def months_since(loyalty_number):
    if loyalty_number not in last_flight_date.index:
        return 999  # never flew in the feature window
    d = last_flight_date[loyalty_number]
    return (CUTOFF.year - d.year) * 12 + (CUTOFF.month - d.month)

behaviour['Months Since Last Flight'] = behaviour['Loyalty Number'].map(months_since)
behaviour[['Loyalty Number', 'Months Since Last Flight']].head()

,Loyalty Number,Months Since Last Flight
0,100018,2
1,100102,0
2,100140,0
3,100214,0
4,100272,0


In [ ]:

recent3 = feat_activity[feat_activity['Date'] > CUTOFF - pd.DateOffset(months=3)]
recent6 = feat_activity[feat_activity['Date'] > CUTOFF - pd.DateOffset(months=6)]

behaviour = behaviour.set_index('Loyalty Number')
behaviour['Flights in last 3 months'] = recent3.groupby('Loyalty Number')['Total Flights'].sum()
behaviour['Flights in last 6 months'] = recent6.groupby('Loyalty Number')['Total Flights'].sum()
behaviour[['Flights in last 3 months', 'Flights in last 6 months']] = (
    behaviour[['Flights in last 3 months', 'Flights in last 6 months']].fillna(0)
)
behaviour = behaviour.reset_index()
behaviour.shape

(16737, 8)

## 3. Loyalty features (status as of the cutoff)

Two eligibility checks :

- **Enrolled after cutoff** → drop. No fair feature history exists for them yet.
- **Cancelled before cutoff** → drop. They've already left before our
  observation point, so there's no "future" left to predict for them.


In [ ]:
loyalty = loyalty.copy()

def cancellation_date(row):
    if row['Cancellation Year'] == 0:
        return pd.NaT
    return pd.Timestamp(year=int(row['Cancellation Year']), month=int(row['Cancellation Month']), day=1)

loyalty['Cancellation Date'] = loyalty.apply(cancellation_date, axis=1)
loyalty['Enrollment Date'] = pd.to_datetime(
    dict(year=loyalty['Enrollment Year'], month=loyalty['Enrollment Month'], day=1)
)

loyalty['Cancelled Before Cutoff'] = loyalty['Cancellation Date'].notna() & (loyalty['Cancellation Date'] <= CUTOFF)
loyalty['Enrolled After Cutoff'] = loyalty['Enrollment Date'] > CUTOFF

loyalty['Promotion Customer'] = (loyalty['Enrollment Type'] == '2018 promotion').astype(int)
loyalty['Tenure'] = (CUTOFF.year - loyalty['Enrollment Year']) * 12 + (CUTOFF.month - loyalty['Enrollment Month'])

print('Cancelled before cutoff:', loyalty['Cancelled Before Cutoff'].sum())
print('Enrolled after cutoff:', loyalty['Enrolled After Cutoff'].sum())

eligible = loyalty[~loyalty['Cancelled Before Cutoff'] & ~loyalty['Enrolled After Cutoff']].copy()
print('Eligible customers for this analysis:', len(eligible))

Cancelled before cutoff: 1707
Enrolled after cutoff: 1331
Eligible customers for this analysis: 13699


## 4. Merge behavioural + loyalty features, build Engagement Score

Customers with no activity rows in the feature window (they simply didn't fly)
get filled with 0s rather than dropped. They are an important part of the data.


In [ ]:
final_dataset = eligible.merge(behaviour, on='Loyalty Number', how='left')

fill_zero_cols = [
    'Total Flights in Lifetime', 'Total Distance', 'Net Reward Points', 'Redeem Ratio',
    'Flights in last 3 months', 'Flights in last 6 months'
]
final_dataset[fill_zero_cols] = final_dataset[fill_zero_cols].fillna(0)
final_dataset['Months Since Last Flight'] = final_dataset['Months Since Last Flight'].fillna(999)

def normalized(s):
    rng = s.max() - s.min()
    return (s - s.min()) / rng if rng != 0 else 0

final_dataset['Engagement Score'] = (
    normalized(final_dataset['CLV']) +
    normalized(final_dataset['Tenure']) +
    normalized(final_dataset['Flights in last 3 months']) +
    normalized(final_dataset['Flights in last 6 months'])
) / 4.0

final_dataset.shape

(13699, 30)

## 5. Churn label (label window only)

Churn = 1 if **either**:
- the customer took **zero flights** between Jul 2018 and Dec 2018, **or**
- the customer's loyalty account was **cancelled** in that same window

Cancellation is not the only form of Churn. Not flying for a long time matters too.


In [ ]:
flights_in_label_window = label_activity.groupby('Loyalty Number')['Total Flights'].sum()
flew_in_label_window = (flights_in_label_window > 0)

cancel_in_label_window = (
    loyalty['Cancellation Date'].notna() &
    (loyalty['Cancellation Date'] > CUTOFF) &
    (loyalty['Cancellation Date'] <= LABEL_END)
)
cancelled_ids = set(loyalty.loc[cancel_in_label_window, 'Loyalty Number'])

final_dataset['Flew In Label Window'] = final_dataset['Loyalty Number'].map(flew_in_label_window).fillna(False)
final_dataset['Cancelled In Label Window'] = final_dataset['Loyalty Number'].isin(cancelled_ids)

final_dataset['Churn'] = (
    (~final_dataset['Flew In Label Window']) | final_dataset['Cancelled In Label Window']
).astype(int)

churn_rate = final_dataset['Churn'].mean() * 100
print(f'Churn rate: {churn_rate:.2f}%')
final_dataset['Churn'].value_counts()

Churn rate: 5.56%


,count
Churn,
0,12938
1,761


## 6. Leakage check
We check if any single column has suspiciously large correlation with the Churn column.


In [ ]:
numeric_cols = final_dataset.select_dtypes(include=[np.number]).columns
corr_with_churn = final_dataset[numeric_cols].corr()['Churn'].sort_values(key=abs, ascending=False)
corr_with_churn

,Churn
Churn,1.000000
Cancellation Year,0.672531
Cancellation Month,0.662141
Months Since Last Flight,0.437052
Total Flights in Lifetime,-0.293316
Total Distance,-0.274485
Net Reward Points,-0.249979
Flights in last 6 months,-0.139323
Promotion Customer,0.127522
Redeem Ratio,-0.084365


## 6b. One more leakage source — raw cancellation date columns

The check above flagged Cancellation Year and Cancellation Month (0.67
correlation with Churn) those raw columns encode when someone cancelled,
which for customers who cancel during the label window is literally part of
how the label was built. They need to be excluded from any modelling feature
set, even though they're useful to keep in the dataframe for reference/audit.

We define the feature list explicitly.

In [ ]:
non_feature_cols = [
    'Loyalty Number', 'Cancellation Year', 'Cancellation Month', 'Cancellation Date',
    'Cancelled In Label Window', 'Cancelled Before Cutoff', 'Enrolled After Cutoff',
    'Churn'
    'Country',
    'Postal Code',
    'Enrollment date'
    # the last three are not sources of leakage but they are not useful for modelling
]
feature_cols = [c for c in final_dataset.columns if c not in non_feature_cols]

print('Modelling features:')
for c in feature_cols:
    print(' -', c)

# We re-run the leakage check restricted to actual modelling features only
final_dataset[feature_cols + ['Churn']].select_dtypes(include=[np.number]).corr()['Churn'].sort_values(key=abs, ascending=False)

Modelling features:
 - Country
 - Province
 - City
 - Postal Code
 - Gender
 - Education
 - Salary
 - Marital Status
 - Loyalty Card
 - CLV
 - Enrollment Type
 - Enrollment Year
 - Enrollment Month
 - Enrollment Date
 - Promotion Customer
 - Tenure
 - Total Flights in Lifetime
 - Total Distance
 - Net Reward Points
 - Redeem Ratio
 - Months Since Last Flight
 - Flights in last 3 months
 - Flights in last 6 months
 - Engagement Score
 - Flew In Label Window


,Churn
Churn,1.000000
Months Since Last Flight,0.437052
Total Flights in Lifetime,-0.293316
Total Distance,-0.274485
Net Reward Points,-0.249979
Flights in last 6 months,-0.139323
Promotion Customer,0.127522
Redeem Ratio,-0.084365
Flights in last 3 months,-0.063871
Engagement Score,-0.062512


'Months Since Last Flight' and the recent-flights features will still show
*some* correlation with churn. That's expected and fine, since they reflect
behaviour up to the cutoff, not the label window itself. What we've removed is
the exact circularity where the same time-point value was used as both feature
and label.

# Saving the dataset


In [ ]:
final_dataset.drop(columns=['Flew In Label Window'], inplace=True)
final_dataset.to_csv('Final_dataset.csv', index=False)
files.download('Final_dataset.csv')
final_dataset.head()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,Loyalty Number,Country,Province,City,Postal Code,Gender,Education,Salary,Marital Status,Loyalty Card,...,Total Flights in Lifetime,Total Distance,Net Reward Points,Redeem Ratio,Months Since Last Flight,Flights in last 3 months,Flights in last 6 months,Engagement Score,Cancelled In Label Window,Churn
0,480934,canada,ontario,toronto,m2z 4k1,female,bachelor,83236.0,married,star,...,24,31092,30770.0,0.010,1,6,6,0.162575,False,0
1,549612,canada,alberta,edmonton,t3g 6y6,male,college,73455.0,divorced,star,...,45,56299,55159.0,0.020,0,11,14,0.225587,False,0
2,608370,canada,ontario,toronto,p1w 1k4,male,college,73455.0,single,star,...,30,42154,41290.0,0.020,0,3,6,0.267894,False,0
3,530508,canada,quebec,hull,j8y 3z5,male,bachelor,103495.0,married,star,...,20,34361,34361.0,0.000,0,7,9,0.236781,False,0
4,193662,canada,yukon,whitehorse,y2k 6r0,male,bachelor,51124.0,married,star,...,71,119399,118545.0,0.007,0,13,19,0.416387,False,0
